In [12]:
pip install thop

In [1]:
import os
import random
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from imblearn.metrics import specificity_score
from mambapy.mamba import Mamba, MambaConfig

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory allocated at start: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Device: cuda
GPU memory allocated at start: 0.00 GB


In [2]:
class BiMambaWrapper(nn.Module):
    """
    Vision Mamba's bidirectional idea, built on mambapy's tested, verified
    Mamba block: run it once forward, once on the reversed sequence, then
    combine. The scan/recurrence math is entirely mambapy's (Blelloch
    parallel scan, numerically verified against the official implementation,
    part of Hugging Face transformers).
    """
    def __init__(self, d_model, n_layers=2, d_state=16):
        super().__init__()
        config = MambaConfig(d_model=d_model, n_layers=n_layers, d_state=d_state)
        self.mamba_fwd = Mamba(config)
        self.mamba_bwd = Mamba(config)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        # x: (B, L, d_model)
        y_fwd = self.mamba_fwd(x)
        y_bwd = torch.flip(self.mamba_bwd(torch.flip(x, dims=[1])), dims=[1])
        return self.norm(y_fwd + y_bwd)

In [3]:
class ROIPatchEmbed(nn.Module):
    """Splits each 64^3 ROI into non-overlapping 8^3 patches -> tokens.
    6 ROIs * 8*8*8 patches = 3072 tokens per subject per modality."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=64):
        super().__init__()
        self.grid_size = roi_size // patch_size
        self.n_tokens = (self.grid_size ** 3) * n_rois
        self.patch_conv = nn.Conv3d(1, d_model, kernel_size=patch_size, stride=patch_size)
        self.pos_embed = nn.Parameter(torch.randn(1, self.n_tokens, d_model) * 0.02)

    def forward(self, rois):
        B, N = rois.shape[0], rois.shape[1]
        toks = [self.patch_conv(rois[:, i]).flatten(2).transpose(1, 2) for i in range(N)]
        return torch.cat(toks, dim=1) + self.pos_embed


class VisionMambaBranch(nn.Module):
    """Patch embed -> bidirectional Mamba -> mean pool."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=64, n_layers=2, d_state=16):
        super().__init__()
        self.patch_embed = ROIPatchEmbed(n_rois, roi_size, patch_size, d_model)
        self.bimamba = BiMambaWrapper(d_model, n_layers, d_state)

    def forward(self, rois):
        tokens = self.patch_embed(rois)
        tokens = self.bimamba(tokens)
        return tokens.mean(dim=1)


class VisionMambaModel(nn.Module):
    """Single-modality model -- use for MRI-only or PET-only."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=64,
                 n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, rois):
        pooled = self.branch(rois)
        return self.classifier(self.dropout(pooled))


class MultimodalVisionMambaModel(nn.Module):
    """Late fusion -- separate MRI/PET branches, concatenated before classifier."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=64,
                 n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.mri_branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.pet_branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model * 2, n_classes)

    def forward(self, mri_rois, pet_rois):
        fused = torch.cat([self.mri_branch(mri_rois), self.pet_branch(pet_rois)], dim=1)
        return self.classifier(self.dropout(fused))

In [4]:
COHORT_CSV    = "D:/mamba_model/thesis_cohort_final.csv"
MRI_CACHE_AUG = "D:/mamba_model/preprocessed_cache_roi64_aug"
PET_CACHE_AUG = "D:/mamba_model/preprocessed_cache_pet_aug"
CKPT_DIR      = "D:/mamba_model/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

df = pd.read_csv(COHORT_CSV)
sessions = df["mri_session"].values
labels   = df["outcome_label"].values

X_tv, X_test, y_tv, y_test = train_test_split(
    sessions, labels, test_size=0.2, random_state=42, stratify=labels
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv
)
session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 126 | Val: 42 | Test: 42


In [5]:
class ROIDataset(Dataset):
    """Single-modality dataset (MRI-only or PET-only)."""
    def __init__(self, sessions, labels, cache_dir, is_mri=True, is_train=False):
        self.samples = []
        self.cache_dir = cache_dir
        for session_id, label in zip(sessions, labels):
            key = session_id if is_mri else session_to_subject[session_id]
            self.samples.append((key, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((key, label, f"aug{seed}"))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        key, label, version = self.samples[idx]
        rois = np.load(f"{self.cache_dir}/{key}_{version}.npy").astype(np.float32)
        return torch.tensor(rois).unsqueeze(1), torch.tensor(label, dtype=torch.long)


class MultimodalROIDataset(Dataset):
    """Pairs matching MRI and PET aug files per subject/seed."""
    def __init__(self, sessions, labels, mri_cache_dir, pet_cache_dir, is_train=False):
        self.samples = []
        self.mri_cache_dir = mri_cache_dir
        self.pet_cache_dir = pet_cache_dir
        for session_id, label in zip(sessions, labels):
            subject_id = session_to_subject[session_id]
            self.samples.append((session_id, subject_id, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((session_id, subject_id, label, f"aug{seed}"))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        mri_key, pet_key, label, version = self.samples[idx]
        mri_rois = np.load(f"{self.mri_cache_dir}/{mri_key}_{version}.npy").astype(np.float32)
        pet_rois = np.load(f"{self.pet_cache_dir}/{pet_key}_{version}.npy").astype(np.float32)
        return (torch.tensor(mri_rois).unsqueeze(1), torch.tensor(pet_rois).unsqueeze(1),
                torch.tensor(label, dtype=torch.long))

In [6]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for rois, labels in loader:
        rois, labels = rois.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(rois)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for rois, labels in loader:
            rois, labels = rois.to(device), labels.to(device)
            outputs = model(rois)
            total_loss += criterion(outputs, labels).item()
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    tpr = recall_score(all_labels, all_preds, zero_division=0)
    tnr = specificity_score(all_labels, all_preds)
    return avg_loss, acc, tpr, tnr

In [7]:
def train_epoch_mm(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for mri_rois, pet_rois, labels in loader:
        mri_rois, pet_rois, labels = mri_rois.to(device), pet_rois.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(mri_rois, pet_rois)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate_mm(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for mri_rois, pet_rois, labels in loader:
            mri_rois, pet_rois, labels = mri_rois.to(device), pet_rois.to(device), labels.to(device)
            outputs = model(mri_rois, pet_rois)
            total_loss += criterion(outputs, labels).item()
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    tpr = recall_score(all_labels, all_preds, zero_division=0)
    tnr = specificity_score(all_labels, all_preds)
    return avg_loss, acc, tpr, tnr

In [8]:
# Captures the computational-efficiency metrics: training time, inference time per scan,
# parameter count, FLOPs.

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def measure_inference_time(model, loader, device, n_batches_to_time=20):
    """Average wall-clock inference time PER SAMPLE, on GPU with proper
    synchronization (so we're timing actual completion, not kernel launch)."""
    model.eval()
    times = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches_to_time:
                break
            if len(batch) == 2:  # single-modality: (rois, labels)
                rois, labels = batch
                rois = rois.to(device)
                batch_size = rois.shape[0]
                if device.type == 'cuda':
                    torch.cuda.synchronize()
                t0 = time.time()
                _ = model(rois)
                if device.type == 'cuda':
                    torch.cuda.synchronize()
                times.append((time.time() - t0) / batch_size)
            else:  # multimodal: (mri_rois, pet_rois, labels)
                mri_rois, pet_rois, labels = batch
                mri_rois, pet_rois = mri_rois.to(device), pet_rois.to(device)
                batch_size = mri_rois.shape[0]
                if device.type == 'cuda':
                    torch.cuda.synchronize()
                t0 = time.time()
                _ = model(mri_rois, pet_rois)
                if device.type == 'cuda':
                    torch.cuda.synchronize()
                times.append((time.time() - t0) / batch_size)
    return np.mean(times), np.std(times)


def try_compute_flops(model, sample_input, is_multimodal=False):
    """
    Attempts FLOPs estimation via thop. NOTE: thop's hooks cover standard
    layers (Conv3d, Linear) reliably but may not fully capture custom
    operations inside mambapy's Mamba scan -- treat this as an approximate
    lower-bound figure, not an exact count. State this caveat if reported
    in your results section.
    """
    try:
        from thop import profile
        model.eval()
        with torch.no_grad():
            if is_multimodal:
                macs, params = profile(model, inputs=sample_input, verbose=False)
            else:
                macs, params = profile(model, inputs=(sample_input,), verbose=False)
        flops = macs * 2  # MACs -> FLOPs
        return flops, True
    except Exception as e:
        print(f"  (FLOPs estimation failed/incomplete: {e})")
        return None, False


def report_efficiency(model, model_name, total_train_time_sec, test_loader, device, is_multimodal=False):
    n_params = count_parameters(model)
    inf_time_mean, inf_time_std = measure_inference_time(model, test_loader, device)

    print(f"\n{'='*60}")
    print(f"EFFICIENCY METRICS -- {model_name}")
    print(f"{'='*60}")
    print(f"Total training time:       {total_train_time_sec/60:.1f} min ({total_train_time_sec:.1f}s)")
    print(f"Trainable parameters:      {n_params:,}")
    print(f"Inference time per scan:   {inf_time_mean*1000:.2f} ms (+/- {inf_time_std*1000:.2f} ms)")

    sample_batch = next(iter(test_loader))
    if is_multimodal:
        sample_input = (sample_batch[0][:1].to(device), sample_batch[1][:1].to(device))
    else:
        sample_input = sample_batch[0][:1].to(device)
    flops, flops_ok = try_compute_flops(model, sample_input, is_multimodal)
    if flops_ok:
        print(f"FLOPs (approx, see note):  {flops/1e9:.2f} GFLOPs")
    print(f"{'='*60}\n")

    return {
        "model_name": model_name,
        "train_time_sec": total_train_time_sec,
        "n_params": n_params,
        "inference_time_ms_mean": inf_time_mean * 1000,
        "inference_time_ms_std": inf_time_std * 1000,
        "flops": flops if flops_ok else None,
    }

In [9]:
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

train_dataset = ROIDataset(X_train, y_train, MRI_CACHE_AUG, is_mri=True, is_train=True)
val_dataset   = ROIDataset(X_val,   y_val,   MRI_CACHE_AUG, is_mri=True, is_train=False)
test_dataset  = ROIDataset(X_test,  y_test,  MRI_CACHE_AUG, is_mri=True, is_train=False)

BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model = VisionMambaModel(d_model=32, n_layers=2, n_classes=2, dropout=0.4).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

best_val_loss = float("inf")
no_improvement = 0
best_epoch = 0
save_path = f"{CKPT_DIR}/vim_mri_only_d32_seed{SEED}.pt"

print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Val Acc':>8} | {'Val TPR':>8} | {'Val TNR':>8} | {'Time':>6}")
print("-" * 75)

total_train_time = 0
for epoch in range(1, 101):
    t0 = time.time()
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, val_tpr, val_tnr = evaluate(model, val_loader, criterion, device)
    scheduler.step(val_loss)
    epoch_time = time.time() - t0
    total_train_time += epoch_time

    print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | "
          f"{val_acc:>8.4f} | {val_tpr:>8.4f} | {val_tnr:>8.4f} | {epoch_time:>5.1f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        no_improvement = 0
        torch.save(model.state_dict(), save_path)
    else:
        no_improvement += 1
        if no_improvement >= 15:
            print(f"Early stopping at epoch {epoch}. Best: {best_epoch}")
            break

model.load_state_dict(torch.load(save_path, weights_only=True))
test_loss, test_acc, test_tpr, test_tnr = evaluate(model, test_loader, criterion, device)
print(f"\nMRI-ONLY TEST: Acc={test_acc*100:.1f}% | TPR={test_tpr*100:.1f}% | TNR={test_tnr*100:.1f}%")

mri_results = {"acc": test_acc, "tpr": test_tpr, "tnr": test_tnr, "best_epoch": best_epoch}
mri_efficiency = report_efficiency(model, "MRI-only (d_model=32)", total_train_time, test_loader, device)

 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
---------------------------------------------------------------------------
     1 |     0.7169 |     0.6988 |   0.5000 |   0.0000 |   1.0000 |  50.3s
     2 |     0.7003 |     0.6985 |   0.5000 |   1.0000 |   0.0000 |  10.5s
     3 |     0.6913 |     0.6933 |   0.5000 |   0.0000 |   1.0000 |  10.4s
     4 |     0.6915 |     0.6932 |   0.5000 |   0.0000 |   1.0000 |  10.1s
     5 |     0.6942 |     0.6898 |   0.5238 |   0.8571 |   0.1905 |   9.8s
     6 |     0.6890 |     0.6890 |   0.5000 |   1.0000 |   0.0000 |   9.8s
     7 |     0.6874 |     0.6925 |   0.5000 |   0.0000 |   1.0000 |  10.1s
     8 |     0.6896 |     0.6861 |   0.6429 |   0.3333 |   0.9524 |  10.4s
     9 |     0.6721 |     0.6849 |   0.5476 |   0.8571 |   0.2381 |  10.1s
    10 |     0.6824 |     0.6854 |   0.5000 |   1.0000 |   0.0000 |  10.1s
    11 |     0.6799 |     0.6943 |   0.5000 |   1.0000 |   0.0000 |  10.4s
    12 |     0.6698 |   

In [10]:
SEED_PET = 42
torch.manual_seed(SEED_PET)
torch.cuda.manual_seed(SEED_PET)
np.random.seed(SEED_PET)
random.seed(SEED_PET)

pet_train_dataset = ROIDataset(X_train, y_train, PET_CACHE_AUG, is_mri=False, is_train=True)
pet_val_dataset   = ROIDataset(X_val,   y_val,   PET_CACHE_AUG, is_mri=False, is_train=False)
pet_test_dataset  = ROIDataset(X_test,  y_test,  PET_CACHE_AUG, is_mri=False, is_train=False)

pet_train_loader = DataLoader(pet_train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
pet_val_loader   = DataLoader(pet_val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
pet_test_loader  = DataLoader(pet_test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

pet_model = VisionMambaModel(d_model=32, n_layers=2, n_classes=2, dropout=0.4).to(device)
pet_criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
pet_optimizer = optim.AdamW(pet_model.parameters(), lr=1e-4, weight_decay=1e-3)
pet_scheduler = optim.lr_scheduler.ReduceLROnPlateau(pet_optimizer, mode='min', factor=0.5, patience=10)

best_val_loss = float("inf")
no_improvement = 0
best_epoch = 0
save_path = f"{CKPT_DIR}/vim_pet_only_d32_seed{SEED_PET}.pt"

print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Val Acc':>8} | {'Val TPR':>8} | {'Val TNR':>8} | {'Time':>6}")
print("-" * 75)

total_train_time = 0
for epoch in range(1, 101):
    t0 = time.time()
    train_loss = train_epoch(pet_model, pet_train_loader, pet_optimizer, pet_criterion, device)
    val_loss, val_acc, val_tpr, val_tnr = evaluate(pet_model, pet_val_loader, pet_criterion, device)
    pet_scheduler.step(val_loss)
    epoch_time = time.time() - t0
    total_train_time += epoch_time

    print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | "
          f"{val_acc:>8.4f} | {val_tpr:>8.4f} | {val_tnr:>8.4f} | {epoch_time:>5.1f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        no_improvement = 0
        torch.save(pet_model.state_dict(), save_path)
    else:
        no_improvement += 1
        if no_improvement >= 15:
            print(f"Early stopping at epoch {epoch}. Best: {best_epoch}")
            break

pet_model.load_state_dict(torch.load(save_path, weights_only=True))
test_loss, test_acc, test_tpr, test_tnr = evaluate(pet_model, pet_test_loader, pet_criterion, device)
print(f"\nPET-ONLY TEST: Acc={test_acc*100:.1f}% | TPR={test_tpr*100:.1f}% | TNR={test_tnr*100:.1f}%")

pet_results = {"acc": test_acc, "tpr": test_tpr, "tnr": test_tnr, "best_epoch": best_epoch}
pet_efficiency = report_efficiency(pet_model, "PET-only (d_model=32)", total_train_time, pet_test_loader, device)

 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
---------------------------------------------------------------------------
     1 |     0.7171 |     0.6955 |   0.5000 |   0.0000 |   1.0000 |  80.1s
     2 |     0.6987 |     0.6963 |   0.5000 |   1.0000 |   0.0000 |  10.6s
     3 |     0.6889 |     0.6897 |   0.5238 |   0.0952 |   0.9524 |  10.2s
     4 |     0.6905 |     0.6891 |   0.5238 |   0.0952 |   0.9524 |   9.9s
     5 |     0.6923 |     0.6868 |   0.6190 |   0.6667 |   0.5714 |   9.9s
     6 |     0.6862 |     0.6867 |   0.4762 |   0.9048 |   0.0476 |   9.9s
     7 |     0.6848 |     0.6900 |   0.5238 |   0.0952 |   0.9524 |   9.9s
     8 |     0.6881 |     0.6841 |   0.5714 |   0.3333 |   0.8095 |  10.2s
     9 |     0.6711 |     0.6837 |   0.5714 |   0.8095 |   0.3333 |  10.0s
    10 |     0.6808 |     0.6853 |   0.5000 |   1.0000 |   0.0000 |  10.0s
    11 |     0.6798 |     0.6936 |   0.5000 |   1.0000 |   0.0000 |   9.8s
    12 |     0.6688 |   

In [13]:
SEED_MM = 42
torch.manual_seed(SEED_MM)
torch.cuda.manual_seed(SEED_MM)
np.random.seed(SEED_MM)
random.seed(SEED_MM)

mm_train_dataset = MultimodalROIDataset(X_train, y_train, MRI_CACHE_AUG, PET_CACHE_AUG, is_train=True)
mm_val_dataset   = MultimodalROIDataset(X_val,   y_val,   MRI_CACHE_AUG, PET_CACHE_AUG, is_train=False)
mm_test_dataset  = MultimodalROIDataset(X_test,  y_test,  MRI_CACHE_AUG, PET_CACHE_AUG, is_train=False)

mm_train_loader = DataLoader(mm_train_dataset, batch_size=4, shuffle=True,  num_workers=0)
mm_val_loader   = DataLoader(mm_val_dataset,   batch_size=4, shuffle=False, num_workers=0)
mm_test_loader  = DataLoader(mm_test_dataset,  batch_size=4, shuffle=False, num_workers=0)

mm_model = MultimodalVisionMambaModel(d_model=32, n_layers=2, n_classes=2, dropout=0.4).to(device)
mm_criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
mm_optimizer = optim.AdamW(mm_model.parameters(), lr=1e-4, weight_decay=1e-3)
mm_scheduler = optim.lr_scheduler.ReduceLROnPlateau(mm_optimizer, mode='min', factor=0.5, patience=10)

best_val_loss = float("inf")
no_improvement = 0
best_epoch = 0
save_path = f"{CKPT_DIR}/vim_multimodal_d32_seed{SEED_MM}.pt"

print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Val Acc':>8} | {'Val TPR':>8} | {'Val TNR':>8} | {'Time':>6}")
print("-" * 75)

total_train_time = 0
for epoch in range(1, 101):
    t0 = time.time()
    train_loss = train_epoch_mm(mm_model, mm_train_loader, mm_optimizer, mm_criterion, device)
    val_loss, val_acc, val_tpr, val_tnr = evaluate_mm(mm_model, mm_val_loader, mm_criterion, device)
    mm_scheduler.step(val_loss)
    epoch_time = time.time() - t0
    total_train_time += epoch_time

    print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | "
          f"{val_acc:>8.4f} | {val_tpr:>8.4f} | {val_tnr:>8.4f} | {epoch_time:>5.1f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        no_improvement = 0
        torch.save(mm_model.state_dict(), save_path)
    else:
        no_improvement += 1
        if no_improvement >= 15:
            print(f"Early stopping at epoch {epoch}. Best: {best_epoch}")
            break

mm_model.load_state_dict(torch.load(save_path, weights_only=True))
test_loss, test_acc, test_tpr, test_tnr = evaluate_mm(mm_model, mm_test_loader, mm_criterion, device)
print(f"\nMULTIMODAL TEST: Acc={test_acc*100:.1f}% | TPR={test_tpr*100:.1f}% | TNR={test_tnr*100:.1f}%")

mm_results = {"acc": test_acc, "tpr": test_tpr, "tnr": test_tnr, "best_epoch": best_epoch}
mm_efficiency = report_efficiency(mm_model, "Multimodal (d_model=32)", total_train_time, mm_test_loader, device, is_multimodal=True)

 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
---------------------------------------------------------------------------
     1 |     0.7069 |     0.6943 |   0.5000 |   0.0000 |   1.0000 |  20.7s
     2 |     0.7136 |     0.6902 |   0.5714 |   0.2857 |   0.8571 |  21.3s
     3 |     0.6939 |     0.6922 |   0.5000 |   1.0000 |   0.0000 |  21.6s
     4 |     0.6921 |     0.6886 |   0.5238 |   0.0952 |   0.9524 |  21.0s
     5 |     0.6879 |     0.6861 |   0.6190 |   0.5714 |   0.6667 |  21.0s
     6 |     0.6798 |     0.6881 |   0.5238 |   0.0952 |   0.9524 |  20.7s
     7 |     0.6886 |     0.6996 |   0.5000 |   1.0000 |   0.0000 |  21.5s
     8 |     0.6787 |     0.6830 |   0.5952 |   0.8095 |   0.3810 |  21.4s
     9 |     0.6740 |     0.6823 |   0.5238 |   0.9048 |   0.1429 |  20.7s
    10 |     0.6707 |     0.7028 |   0.5000 |   1.0000 |   0.0000 |  21.1s
    11 |     0.6746 |     0.6826 |   0.5000 |   1.0000 |   0.0000 |  21.2s
    12 |     0.6608 |   

In [14]:
print(f"{'Model':<25} | {'Acc':>7} | {'TPR':>7} | {'TNR':>7} | {'Params':>10} | {'Train (min)':>11} | {'Inf/scan (ms)':>13}")
print("-" * 95)
print(f"{'MNA-net (Vo et al.)':<25} | {'82.9%':>7} | {'85.7%':>7} | {'80.0%':>7} | {'--':>10} | {'--':>11} | {'--':>13}")
print(f"{'MRI-only':<25} | {mri_results['acc']*100:>6.1f}% | {mri_results['tpr']*100:>6.1f}% | {mri_results['tnr']*100:>6.1f}% | "
      f"{mri_efficiency['n_params']:>10,} | {mri_efficiency['train_time_sec']/60:>10.1f}m | {mri_efficiency['inference_time_ms_mean']:>12.2f}ms")
print(f"{'PET-only':<25} | {pet_results['acc']*100:>6.1f}% | {pet_results['tpr']*100:>6.1f}% | {pet_results['tnr']*100:>6.1f}% | "
      f"{pet_efficiency['n_params']:>10,} | {pet_efficiency['train_time_sec']/60:>10.1f}m | {pet_efficiency['inference_time_ms_mean']:>12.2f}ms")
print(f"{'Multimodal':<25} | {mm_results['acc']*100:>6.1f}% | {mm_results['tpr']*100:>6.1f}% | {mm_results['tnr']*100:>6.1f}% | "
      f"{mm_efficiency['n_params']:>10,} | {mm_efficiency['train_time_sec']/60:>10.1f}m | {mm_efficiency['inference_time_ms_mean']:>12.2f}ms")

Model                     |     Acc |     TPR |     TNR |     Params | Train (min) | Inf/scan (ms)
-----------------------------------------------------------------------------------------------
MNA-net (Vo et al.)       |   82.9% |   85.7% |   80.0% |         -- |          -- |            --
MRI-only                  |   71.4% |   85.7% |   57.1% |    154,658 |        9.3m |         8.34ms
PET-only                  |   69.0% |   76.2% |   61.9% |    154,658 |       10.5m |         8.75ms
Multimodal                |   69.0% |   66.7% |   71.4% |    309,314 |       15.9m |        14.61ms
